# Load neuron mesh and spines with head/neck classification

Copyright (c) 2026 Open Brain Institute

Authors: Marwan Abdellah, Michael W. Reimann

last modified: 06.2026

This notebook demonstrates how to:
1. Find all CellMorphologies that you have access to that were skeletonized from a given EM Dataset
2. Load the neuron mesh of the selected neuron.
3. Load the corresponding `.h5` file containing spines with head/neck data
4. Access head-only and neck-only spine meshes
5. Interactively select and zoom on individual spines using k3d

## Imports and select project

Select the project you want to work with. 
Important: Selection of the project determines which neuron morphologies you have access to!

In [ ]:
from morph_spines import load_morphology_with_spines

import obi_auth
import pylmesh

from entitysdk import Client, types, models
from obi_one import CellMorphologyFromID
from entitysdk.models import EMCellMesh, EMDenseReconstructionDataset, CellMorphology
from obi_notebook. get_projects import get_projects
from obi_notebook.get_environment import get_environment
from ipywidgets import widgets

import logging
loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    logger.setLevel(logging.ERROR)

env_ = get_environment()
token = obi_auth.get_token(environment=env_, auth_mode="daf")
project_context = get_projects(token, env=env_)

## Select EM Dataset to consider
Skeletonized morphologies stem from an electron microscopy dataset.

The following dropdown displays all EM datasets accessible on the OBI platform. Select the one to consider.

**NOTE**: 
For some of the datasets you may not have access to any skeletonized morphologies. In that case, try a different one or run the `Skeletonization` workflow on the main platform.

In [ ]:
client = Client(project_context=project_context, token_manager=token, environment=env_)

em_datasets = client.search_entity(entity_type=models.EMDenseReconstructionDataset).all()

sel_em = widgets.Dropdown(options={dataset.name: dataset for dataset in em_datasets})
display(sel_em)


## Select neuron morphology to consider
Next, you specify the neuron you want to visualize the spines of. You can do this by directly specifying the `ID` of the neuron on the platform. Note that this is *not* what is called the "pt_root_id", but an identifier that is internal to the OBI platform. You can find it by using the "copy ID" button in the "Data" section of the OBI virtual labs.

Alternatively, we will list a dropdown of all skeletonized neurons you have access to that were derived from the selected EM dataset.

In [ ]:
neuron_id = "PASTE ID IN HERE"

sel_nrn = None
if neuron_id == "PASTE ID IN HERE":
    # Find all CellMorphologies from MICrONS
    derivations = client.search_entity(entity_type=models.Derivation, query={
        "derivation_type": types.DerivationType.em_dense_reconstruction_dataset_cell_morphology,
        "used__id": sel_em.value.id
    })
    morphologies = [client.get_entity(entity_id=derivation.generated.id, entity_type=CellMorphology)
                    for derivation in derivations]
    sel_nrn = widgets.Dropdown(options={m.description: m for m in morphologies})
    display(sel_nrn)



## Download the data into the notebook

Here, we access the data from the database and load it.
This can take a few seconds.

In [ ]:
if sel_nrn is not None:
    morphology = CellMorphologyFromID(id_str=str(sel_nrn.value.id))
else:
    morphology = CellMorphologyFromID(id_str=neuron_id)
# Where to place the neuron and mesh
mesh_path = "neuron_mesh.glb"
neuron_path = "neuron_with_spines.h5"

# Load spiny neuron
m = morphology.spiny_morphology(db_client=client, path=neuron_path)
print(f"Spine count: {m.spines.spine_count}")

# Download and load mesh
mesh = morphology.source_mesh_entity(db_client=client)
client.download_file(entity_id=mesh.id, entity_type=models.EMCellMesh, asset_id=mesh.assets[0].id, output_path=mesh_path)
neuron_mesh = pylmesh.load_mesh(mesh_path)

## Interactive spine viewer (k3d)

Use the dropdown to select a spine. The viewer shows the spine head (red), neck (green),
and a cropped region of the neuron mesh (gray) for context.

In [ ]:
import numpy as np
import k3d
import ipywidgets as widgets
from IPython.display import display

# Neuron mesh vertices as point cloud
neuron_pts = (np.vstack([[v.x, v.y, v.z] for v in neuron_mesh.vertices]) * 1E-3).astype(np.float32)
print(f'Neuron point cloud: {len(neuron_pts)} points')

# Create K3D plot
plot = k3d.plot(grid_visible=False, background_color=0xffffff)

# Draw the neuron mesh as point cloud (constant screen-space size)
plot += k3d.points(
    neuron_pts,
    point_size=0.05,
    color=0x88CCEE,
    opacity=0.5,
    shader='gaussian',
)

# State: track spine mesh objects for clean removal
spine_meshes = []

def display_spine(spine_id):
    """Display a spine (head/neck) and focus the camera on it."""
    global spine_meshes, plot

    # Remove previous spine meshes
    for obj in spine_meshes:
        try:
            plot -= obj
        except Exception:
            pass
    spine_meshes.clear()

    neck = m.spines.spine_mesh(spine_id, include_head=False)
    head = m.spines.spine_mesh(spine_id, include_neck=False)

    if len(neck.faces) > 0:
        obj = k3d.mesh(
            neck.vertices.astype(np.float32),
            neck.faces.astype(np.uint32),
            color=0x64c864,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    if len(head.faces) > 0:
        obj = k3d.mesh(
            head.vertices.astype(np.float32),
            head.faces.astype(np.uint32),
            color=0xff6464,
            flat_shading=True,
        )
        plot += obj
        spine_meshes.append(obj)

    # Focus camera on the spine
    full = m.spines.spine_mesh(spine_id)
    center = full.centroid.astype(np.float32)
    radius = float(np.linalg.norm(full.vertices - center, axis=1).max())
    camera_pos = center + np.array([0, 0, 4.0 * radius], dtype=np.float32)
    plot.camera_auto_fit = False
    plot.camera = camera_pos.tolist() + center.tolist() + [0, 1, 0]

# Build list of valid spine indices (those with non-empty meshes)
valid_spines = []
for i in range(m.spines.spine_count):
    try:
        mesh = m.spines.spine_mesh(i)
        if mesh is not None and len(mesh.faces) > 0:
            valid_spines.append(i)
    except Exception:
        pass
print(f'Valid spines: {len(valid_spines)} / {m.spines.spine_count}')

# Dropdown for spine selection
dropdown = widgets.Dropdown(
    options=valid_spines,
    value=valid_spines[0],
    description='Spine #:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px'),
)

def on_change(change):
    if change.get('name') == 'value':
        display_spine(change['new'])

dropdown.observe(on_change, names='value')

# Camera reset button
reset_btn = widgets.Button(description='Reset Camera', button_style='primary')

def reset_camera(_=None):
    plot.camera_reset() 
    plot.camera_auto_fit = True

reset_btn.on_click(reset_camera)

# Camera mode toggle button
cam_btn = widgets.ToggleButton(value=True, description='Ortho', button_style='info', tooltip='Toggle Ortho/Perspective')

# Layout and display
controls = widgets.HBox([dropdown, reset_btn])
display(widgets.VBox([plot, controls]))
# Trigger initial spine focus
dropdown.value = valid_spines[1] if len(valid_spines) > 1 else valid_spines[0]
dropdown.value = valid_spines[0]
